# Lab 15: Repeated games with an LLM agent
**MST 0441: Consumers, Trade and Business Strategy**
*Second in-person block, session 15. About 45 minutes.*

The theory says cooperation survives when the future matters enough:
$$\delta \ge \frac{v^d - v^c}{v^d - v^*}.$$

In this lab you (1) find the threshold numerically, (2) compute how it moves
as the cartel grows, (3) run a tournament of strategies, and (4) play the game
against an LLM agent and compare its play with the theory.

### How this lab works

1. Run `Runtime -> Run all` first. The notebook runs as it stands. Read the
   output, then return to the top.
2. Do the cells marked YOUR TURN. Each asks for a number, a line, or a short
   function. The cells are independent; a wrong answer in one does not affect
   the others.
3. Each YOUR TURN ends with a `check(...)` that reports whether your answer
   matches. Nothing raises an error.
4. The last section, *Work with your assistant*, calls your model from code
   through `ask_model()`. One-time setup: the key guide on Canvas. No key?
   Every prompt is a plain string you can copy into a chat window instead;
   paste the reply where marked. Test each reply in code before accepting it.

Nothing to install. `numpy`, `matplotlib` and `requests` are preinstalled in
Colab.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from fractions import Fraction

def check(name, got, want, tol=1e-9):
    """Friendly checker: prints, never raises."""
    if got is None:
        print(f"  ..  {name}: not filled in yet")
        return False
    try:
        if isinstance(want, (list, tuple, np.ndarray)):
            good = np.allclose(np.asarray(got, dtype=float),
                               np.asarray(want, dtype=float), atol=tol)
        elif isinstance(want, str):
            good = str(got).strip().lower() == want.strip().lower()
        elif isinstance(want, bool):
            good = bool(got) is want
        else:
            good = abs(float(got) - float(want)) < tol
    except Exception as e:
        print(f"  XX  {name}: could not compare ({type(e).__name__}: {e})")
        return False
    print(f"  {'OK ' if good else 'XX '} {name} = {got}" + ("" if good else f"   (expected {want})"))
    return good

print("ready")

## 1. Nora and Petter  (given: just run it)

A15 Problem 1. Weekly profits, in thousands:

| | Cooperate | Defect |
|---|---|---|
| **Cooperate** | 4, 4 | 0, 6 |
| **Defect** | 6, 0 | 2, 2 |

In [ ]:
CC, CD, DC, DD = 4.0, 0.0, 6.0, 2.0

def value_cooperate(delta):
    """Cooperate forever: CC every period."""
    return CC / (1 - delta)

def value_deviate(delta):
    """Defect once (DC), then grim trigger punishes with DD forever."""
    return DC + delta * DD / (1 - delta)

ds = np.linspace(0.01, 0.95, 300)
plt.figure(figsize=(6.4, 4))
plt.plot(ds, [value_cooperate(d) for d in ds], lw=2, label="comply forever")
plt.plot(ds, [value_deviate(d)  for d in ds], lw=2, label="deviate once, then punished")
plt.axvline(0.5, ls=":", color="gray"); plt.text(0.51, 12, "$\\delta^* = 1/2$", color="gray")
plt.xlabel("$\\delta$"); plt.ylabel("present value"); plt.ylim(0, 40)
plt.legend(frameon=False); plt.grid(alpha=.3); plt.show()

## 2. YOUR TURN: find $\delta^*$ without algebra

You solved this by hand and got $\delta^* = 1/2$. Now find it numerically: it is
the $\delta$ at which the two curves cross. Bisection is four lines.

In [ ]:
def critical_delta(lo=0.001, hi=0.999, iters=60):
    """Smallest delta at which cooperating is at least as good as deviating."""
    for _ in range(iters):
        mid = (lo + hi) / 2
        cooperating_is_better = None       # <-- YOUR TURN: one comparison
        if cooperating_is_better is None:
            return None
        if cooperating_is_better:
            hi = mid
        else:
            lo = mid
    return (lo + hi) / 2

d_star = critical_delta()
print("delta* =", d_star)
check("delta*", d_star, 0.5, tol=1e-6)

## 3. YOUR TURN: does collusion get harder with more firms?

A15 Problem 2: $P = 24 - Q$, zero marginal cost, $n$ identical firms.
You derived $\delta^* = 9/17$ for $n = 2$ and $4/7$ for $n = 3$. Fill in the three
profit formulas and let the code do the rest.

$$\pi^c = \frac{144}{n}, \qquad
  \pi^* = \frac{576}{(n+1)^2}, \qquad
  \pi^d = \frac{36(n+1)^2}{n^2}.$$

In [ ]:
def cartel_delta(n):
    pi_c = None      # <-- YOUR TURN: cartel profit per firm
    pi_n = None      # <-- YOUR TURN: one-shot Cournot profit per firm
    pi_d = None      # <-- YOUR TURN: best one-period deviation profit
    if None in (pi_c, pi_n, pi_d):
        return None
    return (pi_d - pi_c) / (pi_d - pi_n)

for n in (2, 3, 4, 10):
    print(f"n = {n:>2}:  delta* = {cartel_delta(n)}")

check("delta* at n = 2", cartel_delta(2), 9/17)
check("delta* at n = 3", cartel_delta(3), 4/7)

In [ ]:
ns = np.arange(2, 16)
vals = [cartel_delta(n) for n in ns]
if all(v is not None for v in vals):
    plt.figure(figsize=(6.2, 3.6))
    plt.plot(ns, vals, "o-", lw=2)
    plt.axhline(1.0, ls=":", color="red")
    plt.xlabel("number of firms, n"); plt.ylabel("critical $\\delta^*$")
    plt.title("Collusion needs a more patient cartel as it grows")
    plt.grid(alpha=.3); plt.show()
    print("As n grows, delta* rises toward 1: eventually NO discount factor "
          "below 1 sustains the cartel.")

## 4. A tournament of strategies  (given, then YOUR TURN)

A strategy is just a function: it sees the history and returns `"C"` or `"D"`.

In [ ]:
def always_cooperate(my_hist, their_hist): return "C"
def always_defect(my_hist, their_hist):    return "D"
def grim(my_hist, their_hist):             return "D" if "D" in their_hist else "C"

def tit_for_tat(my_hist, their_hist):
    """Cooperate first, then copy whatever they did last."""
    return "C" if not their_hist else their_hist[-1]

PAY = {("C","C"): (CC, CC), ("C","D"): (CD, DC),
       ("D","C"): (DC, CD), ("D","D"): (DD, DD)}

def play(s1, s2, delta=0.9, rounds=60):
    h1, h2, v1, v2 = [], [], 0.0, 0.0
    for t in range(rounds):
        a1, a2 = s1(h1, h2), s2(h2, h1)
        p1, p2 = PAY[(a1, a2)]
        v1 += (delta**t) * p1; v2 += (delta**t) * p2
        h1.append(a1); h2.append(a2)
    return v1, v2, "".join(h1), "".join(h2)

strategies = {"always-C": always_cooperate, "always-D": always_defect,
              "grim": grim, "tit-for-tat": tit_for_tat}

print(f"{'':<12}" + "".join(f"{n:>12}" for n in strategies))
for n1, s1 in strategies.items():
    row = f"{n1:<12}"
    for n2, s2 in strategies.items():
        row += f"{play(s1, s2)[0]:>12.1f}"
    print(row)

`tit-for-tat` appears only here in the course. It never beats its opponent in
any single match and still scores well overall.

In [ ]:
# <-- YOUR TURN: at delta = 0.3 (an impatient world), which strategy earns most
#     against the field? Predict first, then run.
predicted_winner = None       # e.g. "grim"

totals = {n: sum(play(s, s2, delta=0.3)[0] for s2 in strategies.values())
          for n, s in strategies.items()}
print(sorted(totals.items(), key=lambda kv: -kv[1]))
check("winner at delta = 0.3", predicted_winner, max(totals, key=totals.get))

---
## Your session 15 tutor

The last one. It carries the repeated-games model and the collusion
problems, and it can run your threshold and tournament code.

In [ ]:
import json

# Your model, as a function. (The TUTOR notebook has its own ask() that
# talks to the course tutor; this is a different function, and both can
# live in one notebook.) Works with a free Gemini key (Google AI Studio)
# Course route: an OpenRouter key running Mistral. One-time setup: key icon
# in the left sidebar -> add secret OPENROUTER_API_KEY, allow notebook access.
# (A free Gemini key under GOOGLE_API_KEY also works, as a fallback.)
# Never paste a key into a cell.

# ---- WHICH MODEL THIS STUDENT GETS --------------------------------------
# INSTRUCTOR: set EXPERIMENT below. "same" gives everyone MODEL_A.
# "split" sends a random half to MODEL_A and the other half to MODEL_B,
# assigned from the student id, so the same student always lands in the
# same arm however many times they re-run the notebook.
EXPERIMENT = "same"                                  # "same" or "split"
MODEL_A    = "mistralai/mistral-medium-3-5"          # strong at tool calling
MODEL_B    = "mistralai/mistral-small-2603"          # smaller, cheaper
GEMINI_MODEL = "gemini-3.6-flash"                    # fallback route only

ACTIVE_MODEL = MODEL_A

def assign_model(student_id=""):
    """Pick this student's model. Deterministic: same id -> same arm."""
    global ACTIVE_MODEL
    if EXPERIMENT == "split" and student_id.strip():
        import hashlib
        digest = hashlib.sha256(student_id.strip().lower().encode()).hexdigest()
        ACTIVE_MODEL = MODEL_A if int(digest, 16) % 2 == 0 else MODEL_B
    else:
        ACTIVE_MODEL = MODEL_A
    return ACTIVE_MODEL

def _get_secret(name):
    """Colab Secrets first; an environment variable as the fallback, so the
    instructor can test outside Colab. Never a value pasted into a cell."""
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    import os
    return os.environ.get(name)

def _credentials():
    for secret, base, model in [
        ("OPENROUTER_API_KEY",
         "https://openrouter.ai/api/v1",
         ACTIVE_MODEL),
        ("GOOGLE_API_KEY",
         "https://generativelanguage.googleapis.com/v1beta/openai",
         GEMINI_MODEL),
    ]:
        key = _get_secret(secret)
        if key:
            return base, key, model
    return None

def _chat(messages, tools=None):
    """One HTTP call to the model. Returns its reply message, or None."""
    creds = _credentials()
    if creds is None:
        print("(no API key found -- see the key guide on Canvas)")
        return None
    base, key, model = creds
    payload = {"model": model, "messages": messages}
    if tools:
        payload["tools"] = tools
        payload["tool_choice"] = "auto"
    try:
        import requests
        r = requests.post(base + "/chat/completions",
                          headers={"Authorization": "Bearer " + key},
                          json=payload, timeout=90)
        r.raise_for_status()
        return r.json()["choices"][0]["message"]
    except Exception as e:
        print(f"(the call failed: {type(e).__name__}: {e})")
        return None


def ask_model(prompt, system=None):
    """One plain call: text in, text out. No tools, no loop."""
    messages = ([{"role": "system", "content": system}] if system else [])
    messages.append({"role": "user", "content": prompt})
    reply = _chat(messages)
    return None if reply is None else reply.get("content")


# ---- THE AGENT ----------------------------------------------------------
# A model on its own only writes text. An agent is a model plus TOOLS plus
# a LOOP. Read Conversation.say() once: it is the whole idea, in 20 lines.

class Conversation:
    """An agent you can talk to. It remembers the exchange, and it can call
    your Python functions in the middle of answering -- or ask you a
    question first and wait for your reply."""

    def __init__(self, tools=None, registry=None, system=None, show=True):
        self.tools, self.registry = tools, registry or {}
        self.show = show
        self.messages = [{"role": "system", "content": system}] if system else []

    def say(self, text, max_steps=6):
        """Say something to the agent. Returns its reply, or None with no key."""
        self.messages.append({"role": "user", "content": text})
        for _ in range(max_steps):
            reply = _chat(self.messages, self.tools)
            if reply is None:
                return None
            self.messages.append(reply)
            calls = reply.get("tool_calls")
            if not calls:                       # no tool wanted: it is answering
                return reply.get("content")
            for call in calls:                  # it asked; WE run the function
                name = call["function"]["name"]
                args = json.loads(call["function"]["arguments"] or "{}")
                result = self.registry[name](**args)
                if self.show:
                    print(f"   [agent ran {name}({args})]")
                self.messages.append({"role": "tool", "tool_call_id": call["id"],
                                      "content": json.dumps(result)})
        return "(gave up: too many steps)"


def run_agent(question, tools, registry, max_steps=6, show=True):
    """One-shot version: ask once, get the answer. A Conversation of length 1."""
    return Conversation(tools, registry, show=show).say(question, max_steps)


# ---- THIS SESSION'S TUTOR ----------------------------------------------
# The rules are the same in every lab. What changes is the MODEL the agent
# is teaching and the PROBLEM SET it can look up -- both handed in below.

TUTOR_RULES = """You are a teaching assistant in a first-year microeconomics
course. Rules you always follow:
1. Explain the METHOD before any numbers, in the order the course teaches it.
2. Never invent numbers. Get them by calling your tools.
3. If a parameter you need has not been given, ASK for it and stop. Do not
   assume a value.
4. When you report a result, say in plain words what it means economically,
   including what a high and a low value of the key parameter would imply.
5. Asked about a problem set question, call get_problem FIRST so you work from
   the real wording. Explain the method and the setup. Do NOT give the final
   numbers: leave those to the student. If they show you an answer, check it
   with your tools and say only whether it is right and which step failed.
6. Asked to test the student, ask ONE question at a time and wait. Say whether
   the reply is right before asking the next. Test understanding rather than
   recall: ask why something holds, or what would change if a parameter moved.
7. Be brief. Six sentences at most."""


def make_tutor(session, model_text, problems, tools, registry, show=True):
    """Build this session's tutor: shared rules, this session's knowledge."""
    def get_problem(problem_id):
        return problems.get(problem_id,
                            "no problem " + str(problem_id) + " in this session")

    reg = dict(registry)
    reg["get_problem"] = get_problem
    tls = list(tools) + [{
        "type": "function",
        "function": {
            "name": "get_problem",
            "description": ("Fetch the exact wording of a problem from THIS "
                            "session's problem set, so you work from the real "
                            "question rather than one you imagined. Valid ids: "
                            + ", ".join(sorted(problems)) + "."),
            "parameters": {"type": "object", "properties": {
                "problem_id": {"type": "string", "enum": sorted(problems)}},
                "required": ["problem_id"]}}}]

    system = (TUTOR_RULES + """

THE MODEL YOU ARE TEACHING IN THIS SESSION:
""" + model_text + """

The student is working through a lab on exactly this material, and has the
problem set open beside them.""")
    return Conversation(tls, reg, system=system, show=show)


print("ask_model() and run_agent() ready")

In [ ]:
#@title Session 15 knowledge (double-click if you want to read it)
SESSION15_MODEL = """Repeated games and collusion. In a one-shot prisoner's
dilemma defection is dominant. Repeated infinitely (or with an uncertain end),
cooperation can be sustained by a GRIM TRIGGER: cooperate until someone
defects, then defect forever. Cooperating forever is worth v^c/(1-delta);
deviating pays v^d today and then the punishment v* forever, so cooperation
survives when delta >= (v^d - v^c)/(v^d - v*). That threshold delta* rises
with the size of the cartel: with n firms each gets a smaller share of the
collusive pie but can still steal the whole market for one period, so bigger
cartels need more patience, and beyond some n no delta below 1 works. This is
why competition authorities care about concentration. A FINITELY repeated game
with a known end unravels by backward induction to defection in every round --
though real players, and language models, frequently fail to unravel."""

PROBLEMS15 = {
    "A15.1": ("Nora and Petter play a repeated pricing game. Per-period "
              "profits: both cooperate (4,4); one defects (6,0); both defect "
              "(2,2). Find the critical discount factor delta* at which grim "
              "trigger sustains cooperation, and interpret it."),
    "A15.2": ("The Nordic gravel market: P = 24 - Q, zero marginal cost, n "
              "identical firms. Derive collusive profit per firm, one-shot "
              "Cournot profit, and the best one-period deviation profit; then "
              "find delta*(n) and show it rises with n. Discuss what this "
              "implies for competition policy."),
}

In [ ]:
def threshold_tool(v_cooperate, v_deviate, v_punish):
    """The critical discount factor for grim trigger."""
    d = (v_deviate - v_cooperate) / (v_deviate - v_punish)
    return {"delta_star": round(d, 6),
            "sustainable_if": "delta >= delta_star",
            "note": "compares one period of gain against a permanent punishment"}

def cartel_tool(n):
    """Critical delta for an n-firm cartel in the A15.2 linear-demand market."""
    d = cartel_delta(n)
    return {"n": n, "delta_star": None if d is None else round(d, 6)}

TOOLS15 = [
    {"type": "function", "function": {
        "name": "threshold_tool",
        "description": ("Critical discount factor delta* for sustaining "
                        "cooperation under grim trigger. Pass the per-period "
                        "payoff from cooperating, from deviating once, and "
                        "from the punishment phase."),
        "parameters": {"type": "object", "properties": {
            "v_cooperate": {"type": "number", "description": "payoff each period if both cooperate"},
            "v_deviate": {"type": "number", "description": "payoff in the period you defect"},
            "v_punish": {"type": "number", "description": "payoff each period once punishment starts"}},
            "required": ["v_cooperate", "v_deviate", "v_punish"]}}},
    {"type": "function", "function": {
        "name": "cartel_tool",
        "description": ("Critical discount factor for an n-firm cartel in the "
                        "A15.2 market (P = 24 - Q, zero marginal cost). Use it "
                        "to show how delta* moves as the cartel grows. Needs "
                        "the student's cartel_delta() to be filled in first."),
        "parameters": {"type": "object", "properties": {
            "n": {"type": "integer", "description": "number of firms in the cartel"}},
            "required": ["n"]}}},
]

tutor15 = make_tutor(15, SESSION15_MODEL, PROBLEMS15, TOOLS15,
                     {"threshold_tool": threshold_tool, "cartel_tool": cartel_tool})
print("session 15 tutor ready")

### Ask it the competition-policy question

A15.2(h) asks what the model implies for policy. Make the agent get there
from the numbers rather than from a slogan.

In [ ]:
print(tutor15.say("Using your tools on the A15.2 market, explain how the "
                  "critical discount factor changes as the cartel grows from "
                  "2 to 10 firms, and what that implies for merger policy.")
      or "(no key -- see the key guide)")

In [ ]:
print(tutor15.say("Now test whether I understood why delta* rises with n. "
                  "One question, and wait for my answer.") or "(no key)")

In [ ]:
my_reply = ""      # <-- YOUR TURN: answer in your own words
print(tutor15.say(my_reply) if my_reply else "answer the question above first")

### The agent as a player

The tutor explains the theory. This last part does something different:
it puts a model *inside* the game as an opponent, so you can compare what
the equilibrium says with what it actually does.

### The agent, automated

The same game, but now the model plays inside your loop, against a strategy
function from the tournament. Twelve lines: build the history into the
prompt, send it, parse one character back. This is the machine the whole
thread has been building toward. Change `me` to any strategy from section 4,
or raise `rounds`, and rerun.

In [ ]:
RULES = """We are playing a repeated game. Each round we both simultaneously
choose C or D. Payoffs to you: both C -> 4; you C, I D -> 0; you D, I C -> 6;
both D -> 2. Your goal is to maximize your own total over all rounds.
Reply with exactly one character: C or D."""

def llm_move(its_hist, my_hist):
    """One call: the full history is the context, one character comes back."""
    hist = "\n".join(f"round {t+1}: you {a}, me {b}"
                      for t, (a, b) in enumerate(zip(its_hist, my_hist)))
    r = ask_model(RULES + ("\nHistory so far:\n" + hist if hist else "\nFirst round."))
    if r:
        c = r.strip().upper()[:1]
        if c in "CD":
            return c
    return None

me = tit_for_tat      # <-- YOUR TURN: pick any strategy function from section 4
rounds = 8

mine, its = [], []
for t in range(rounds):
    b = llm_move(its, mine)
    if b is None:
        print("(no key: the six-round chat version above is the same "
              "experiment at hand speed)")
        break
    a = me(mine, its)
    mine.append(a); its.append(b)

if its:
    v_me = sum(PAY[(a, b)][0] for a, b in zip(mine, its))
    v_ag = sum(PAY[(a, b)][1] for a, b in zip(mine, its))
    print("me   :", "".join(mine), f"  total {v_me:g}")
    print("agent:", "".join(its),  f"  total {v_ag:g}")
    print("\nDoes its play look like grim, tit-for-tat, or neither? "
          "Compare with the tournament table.")

---
## Take away

* $\delta^*$ can be found by root-finding as well as algebra; the two
  agreeing at $1/2$ checks both.
* $\delta^*$ rises with $n$: bigger cartels need more patience, which is why
  competition authorities care about concentration.
* Strategies are functions. `tit-for-tat` never wins a match and still does well.
* An LLM agent often cooperates where the equilibrium says defect, including
  when the horizon is finite and known. That gap matters whenever such a
  system acts as a negotiator, a pricing agent, or a counterparty.